
# Simon's Algorithm – 5-bit Implementation (s = '10110')

**Quantum Computing Assignment 2 — Parts B & C**

This notebook implements Simon's algorithm for the 5-bit secret string **s = '10110'**.

- **Part B** — Build and run the quantum circuit (≥ 2000 shots).
- **Part C** — Classical post-processing: solve $Us = 0 \pmod{2}$ to recover $s$.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

print('Imports OK')

---

## Part B — Quantum Circuit

### B.1  Chosen Secret String

In [ ]:
SECRET = '10110'
n = len(SECRET)
print(f"Secret string  : s = '{SECRET}'")
print(f"Number of bits : n = {n}")
print(f"Qubits required: {2*n}  ({n} input + {n} ancilla)")

### B.2  Oracle

For $s = 10110$ we have $s = (s_0, s_1, s_2, s_3, s_4) = (1, 0, 1, 1, 0)$.

The canonical position $k = 0$ (index of the first `'1'` in $s$).  
The oracle implements:

$$f(x_0, x_1, x_2, x_3, x_4) = (0,\; x_1,\; x_2 \oplus x_0,\; x_3 \oplus x_0,\; x_4)$$

**Correctness check** — compute $f(x \oplus s)$ where $x \oplus s = (x_0 \oplus 1, x_1, x_2 \oplus 1, x_3 \oplus 1, x_4)$:

$$f(x \oplus s) = \bigl(0,\; x_1,\; (x_2\oplus1)\oplus(x_0\oplus1),\; (x_3\oplus1)\oplus(x_0\oplus1),\; x_4\bigr)
               = \bigl(0,\; x_1,\; x_2\oplus x_0,\; x_3\oplus x_0,\; x_4\bigr) = f(x). \checkmark$$

**Circuit gates** (input qubits 0–4, output qubits 5–9):

| Gate | Output qubit | Effect |
|------|-------------|--------|
| CNOT(1, 6) | out₁ | $\leftarrow x_1$ |
| CNOT(2, 7) | out₂ | $\leftarrow x_2$ |
| CNOT(0, 7) | out₂ | $\leftarrow x_2 \oplus x_0$ |
| CNOT(3, 8) | out₃ | $\leftarrow x_3$ |
| CNOT(0, 8) | out₃ | $\leftarrow x_3 \oplus x_0$ |
| CNOT(4, 9) | out₄ | $\leftarrow x_4$ |

Qubit 5 (out₀) stays $|0\rangle$.

In [ ]:
def simon_oracle(s: str) -> QuantumCircuit:
    """
    Valid 2-to-1 Simon oracle.
    k = index of first '1' in s.
      out[i] = 0          for i == k
      out[i] = x[i]^x[k] for s[i]='1', i != k
      out[i] = x[i]       for s[i]='0'
    """
    n = len(s)
    qc = QuantumCircuit(2 * n, name='Oracle')
    k = s.index('1')                    # canonical position
    for i in range(n):
        if i != k:
            qc.cx(i, n + i)             # copy all bits except k
    for i in range(n):
        if s[i] == '1' and i != k:
            qc.cx(k, n + i)             # XOR those output bits with x[k]
    return qc


oracle = simon_oracle(SECRET)
print(f"Oracle for s = '{SECRET}':")
print(oracle.draw(output='text'))

### B.3  Full Circuit

In [ ]:
def simon_circuit(s: str) -> QuantumCircuit:
    n = len(s)
    qc = QuantumCircuit(2 * n, n)
    qc.h(range(n))                          # 1st Hadamard
    qc.barrier(label='H₁')
    qc.compose(simon_oracle(s), inplace=True)  # oracle
    qc.barrier(label='Oracle')
    qc.h(range(n))                          # 2nd Hadamard
    qc.barrier(label='H₂')
    qc.measure(range(n), range(n))          # measure input register
    return qc


qc_5 = simon_circuit(SECRET)
print(f"Full Simon circuit for s = '{SECRET}':")
print(qc_5.draw(output='text'))
print(f"\nCircuit depth  : {qc_5.depth()}")
print(f"Total gates    : {qc_5.size()}")

### B.4  Simulation

In [ ]:
SHOTS = 2000
simulator = AerSimulator()

job = simulator.run(transpile(qc_5, simulator), shots=SHOTS)
raw_counts = job.result().get_counts()

# Reverse bit strings so that index 0 is the leftmost character
counts = {k[::-1]: v for k, v in raw_counts.items()}
counts = dict(sorted(counts.items(), key=lambda x: -x[1]))

print(f"Simulation complete. Total shots: {SHOTS}")
print(f"Distinct outcomes : {len(counts)}  (expected 2^(n-1) = {2**(n-1)})")

### B.5  Measurement Outcomes

In [ ]:
non_trivial = {u: c for u, c in counts.items() if u != '0' * n}

print(f"All measured bit strings u (excluding trivial 00000):\n")
print(f"  {'u':12s} {'Count':>6s}   u·s (mod 2)")
print("  " + "-" * 38)
for u, cnt in non_trivial.items():
    dot = sum(int(u[i]) * int(SECRET[i]) for i in range(n)) % 2
    mark = '  ← VIOLATION' if dot != 0 else ''
    print(f"  {u:12s} {cnt:6d}   {dot}{mark}")

all_valid = all(
    sum(int(u[i]) * int(SECRET[i]) for i in range(n)) % 2 == 0
    for u in non_trivial
)
print(f"\nAll outcomes satisfy u·s = 0 (mod 2): {all_valid}")

In [ ]:
plot_histogram(counts, title=f"Simon's Algorithm: s = '{SECRET}'  ({SHOTS} shots)")
plt.tight_layout()
plt.show()

---

## Part C — Classical Post-Processing

### C.1  System of Linear Equations

Each measured outcome $u^{(i)}$ gives one equation over $\mathbb{F}_2$:

$$u^{(i)} \cdot s = \sum_{j=0}^{n-1} u^{(i)}_j \, s_j = 0 \pmod{2}.$$

Stacking all equations gives the matrix system $\mathbf{U}\, s = \mathbf{0} \pmod{2}$.

In [ ]:
u_list = [u for u in non_trivial]   # collect non-trivial outcomes

print("Linear system  U s = 0  (mod 2):")
print()
vars_ = [f's{j}' for j in range(n)]
print(f"  {'Outcome':<12s}   Equation")
print("  " + "-" * 50)
for u in u_list:
    terms = ' + '.join(vars_[j] for j in range(n) if u[j] == '1') or '0'
    print(f"  {u:<12s}   {terms} = 0")

print(f"\nMatrix U  ({len(u_list)} rows × {n} columns):")
U = [[int(c) for c in u] for u in u_list]
for row in U:
    print('  ', row)

### C.2  GF(2) Gaussian Elimination

In [ ]:
def gf2_rref(matrix):
    """Row-reduce a binary matrix to RREF over GF(2)."""
    M = [row[:] for row in matrix]
    n_rows, n_cols = len(M), len(M[0])
    pivot_row = 0
    pivots = []
    for col in range(n_cols):
        found = next((r for r in range(pivot_row, n_rows) if M[r][col] == 1), None)
        if found is None:
            continue
        M[pivot_row], M[found] = M[found], M[pivot_row]
        pivots.append(col)
        for r in range(n_rows):
            if r != pivot_row and M[r][col] == 1:
                M[r] = [M[r][j] ^ M[pivot_row][j] for j in range(n_cols)]
        pivot_row += 1
    free = [c for c in range(n_cols) if c not in pivots]
    return M, pivots, free


rref, pivots, free_cols = gf2_rref(U)

print(f"Pivot columns : {pivots}")
print(f"Free  columns : {free_cols}")
print()
print("RREF (non-zero rows):")
for row in rref:
    if any(row):
        print('  ', row)

### C.3  Back-Substitution — Recovering s

In [ ]:
solutions = []
for fc in free_cols:
    s_vec = [0] * n
    s_vec[fc] = 1                              # free variable = 1
    for idx, pc in enumerate(pivots):
        s_vec[pc] = rref[idx][fc]              # back-substitute pivot
    candidate = ''.join(map(str, s_vec))
    if '1' in candidate:
        solutions.append(candidate)

print("Non-trivial solution(s):")
for sol in solutions:
    print(f"  s = '{sol}'")
print()
print(f"True secret string   : s = '{SECRET}'")
print(f"Strings match        : {solutions == [SECRET] or SECRET in solutions}")

### C.4  Verification

In [ ]:
print("Verification: checking u · s = 0 (mod 2) for every observed u:\n")
print(f"  {'u':12s} u·s (mod 2)")
print("  " + "-" * 28)
recovered = solutions[0]
all_ok = True
for u in u_list:
    dot = sum(int(u[i]) * int(recovered[i]) for i in range(n)) % 2
    if dot != 0:
        all_ok = False
    print(f"  {u:<12s} {dot}")

print()
print(f"All equations satisfied: {all_ok}")
print()
print("═" * 55)
print(f"  Recovered secret string: s = '{recovered}'")
print(f"  True secret string     : s = '{SECRET}'")
print(f"  Recovery successful    : {recovered == SECRET}")
print("═" * 55)